# Image Calibration

Calibrate a **Phase Light Modulator (PLM)** by displaying known phase levels on the PLM
and capturing the resulting interference patterns with a Thorlabs scientific camera.

**Workflow:**
1. Initialize the PLM controller
2. Define the phase-shift analysis function
3. Capture calibration images across all phase levels
4. Plot and analyze the phase response
5. Cleanup


In [3]:
import numpy as np
from PLMController import PLMController
import time
from PIL import Image
import os
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.fft import rfft, rfftfreq
import glob
from thorlabs_tsi_sdk.tl_camera import TLCameraSDK, OPERATION_MODE

---
## 1. PLM Initialization

Define the `initialize_plm()` function. This sets up the connection to the PLM controller,
configures the display (DisplayPort), uploads a 16-level lookup table and a binary phase map.


In [ ]:
def initialize_plm():
    """
    PLM Initializer. Starts the Phase-only Light Modulator sets up the connections, initializes the UI, and defines the lookup table.

    Always check that the displays match the width and height of the PLM in the display settings to the defined values below.
    Define the number of frames you want to enable for the PLM each frame can support 24 RGB holograms.
    """
    MAX_FRAMES = 1
    N = 1358   # PLM width
    M = 800    # PLM height
    # x0 and y0 control where the menu appears.
    x0 = 1920
    y0 = 0

    dll_path = r'..\\bin\\plmctrl.dll'

    plm = PLMController(MAX_FRAMES, N, M, dll_path, x0, y0)

    plm.open()

    source = 0 # Parallel RGB
    port_width = 1
    plm.set_source(source, port_width)
    plm.set_port_swap(0,0)
    # plm.set_port_swap(1,0) # Only use when working with Display Port
    connection_type = 2 # 1 = HDMI, 2 = Display Port.
    plm.set_pixel_mode(connection_type)
    time.sleep(5) # It's important to pause between commands to setup the connection.
    plm.set_connection_type(connection_type)
    time.sleep(10)
    plm.set_video_pattern_mode()
    time.sleep(10)
    play_mode = 1 # 0 = Play Once, 1 = Continuous (Repeat mode)
    plm.update_lut(play_mode, connection_type)

    plm.set_windowed(True)
    plm.start_ui()

    phase_levels = np.array([
        0.000, 0.0100, 0.0205, 0.0422, 0.0560, 0.0727, 0.1131,
        0.1734, 0.3426, 0.3707, 0.4228, 0.4916, 0.5994, 0.6671,
        0.7970, 0.9375, 1.000
    ], dtype=np.float32)

    plm.set_lookup_table(phase_levels)

    phase_map = np.array([
        [0, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [1, 1, 0, 0],
        [0, 0, 1, 0], [1, 0, 1, 0], [0, 1, 1, 0], [1, 1, 1, 0],
        [0, 0, 0, 1], [1, 0, 0, 1], [0, 1, 0, 1], [1, 1, 0, 1],
        [0, 0, 1, 1], [1, 0, 1, 1], [0, 1, 1, 1], [1, 1, 1, 1],
    ], dtype=np.int32)

    phase_map_order = (12, 8, 4, 14, 0, 6, 10, 2, 13, 5, 9, 1, 15, 7, 11, 3)
    phase_map = phase_map[phase_map_order, :]
    plm.set_phase_map(phase_map)
    return phase_levels , M, N, plm



Run the initialization to obtain the PLM controller instance (`plm`), together with
the phase levels, width (`N`) and height (`M`).


In [ ]:
phase_levels, M, N, plm = initialize_plm()

Setting pixel mode to DisplayPort
Setting connection to 2
Setting Video Pattern Mode
Updating bit lookup-table


---
## 2. Phase Shift Analysis Function

Define `calculate_true_phase_shift()` which uses the analytic signal method
(Hilbert transform via FFT) to measure the phase shift from an interference pattern image.
This approach avoids local minima problems associated with `curve_fit`.


In [ ]:
def calculate_true_phase_shift(image_path: str) -> float:
    """
    Calculates the true phase shift in radians using the analytic signal method
    (Hilbert Transform via FFT), avoiding local minima from curve_fit.
    """
    # 1. Load the image in grayscale
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")
    
    height, width = img.shape
    margin = int(width * 0.1)
    center = width // 2
    
    # 2. Obtain averaged 1D profiles (vertical axis)
    profile_left = np.mean(img[:, margin : center - margin], axis=1)
    profile_right = np.mean(img[:, center + margin : width - margin], axis=1)
    
    # Subtract the mean (DC component) to center the oscillation at zero
    sig_l = profile_left - np.mean(profile_left)
    sig_r = profile_right - np.mean(profile_right)
    
    # 3. Transform to the frequency domain (1D FFT)
    fft_l = np.fft.fft(sig_l)
    fft_r = np.fft.fft(sig_r)
    
    # 4. Find the spatial frequency of the fringe
    # Search for the peak in the first half (positive frequencies)
    peak_frequency = np.argmax(np.abs(fft_l[:height // 2]))
    
    # 5. Create a band-pass filter to isolate only the fringe contribution
    # This removes high-frequency speckle noise and background variations
    filt = np.zeros(height)
    bandwidth = 3 # Pixel window around the peak
    filt[peak_frequency - bandwidth : peak_frequency + bandwidth + 1] = 1
    
    # 6. Obtain the analytic signal (analogous to the Hilbert Transform)
    # Keep only the filtered positive frequencies and multiply by 2
    f_left_filtered = fft_l * filt * 2
    f_right_filtered = fft_r * filt * 2
    
    # Transform back to the spatial domain
    analytic_left = np.fft.ifft(f_left_filtered)
    analytic_right = np.fft.ifft(f_right_filtered)
    
    # 7. Extract the continuous spatial phase from each side
    phase_left = np.unwrap(np.angle(analytic_left))
    phase_right = np.unwrap(np.angle(analytic_right))
    
    # 8. Calculate the average phase difference in the central region
    # Average the center region to avoid FFT edge effects
    center_y = slice(height // 4, 3 * height // 4)
    continuous_phase_diff = phase_right[center_y] - phase_left[center_y]
    
    # Average the real phase shift and map to range [0, 2*pi]
    mean_shift = np.mean(continuous_phase_diff)
    final_shift = (mean_shift + 2 * np.pi) % (2 * np.pi)
    
    return float(final_shift)

---
## 3. Camera Calibration

Set up the Thorlabs camera, then cycle through all 16 phase levels. For each level:
- Display the phase pattern on the PLM
- Trigger the camera to capture an image
- Save the image as a PNG

After capturing all images, the measured phase shift is plotted against the level index.


In [ ]:
DLL_DIR = os.path.abspath(
    os.path.join("..", "..", "scientific_camera_interfaces_windows-2.1",
                 "Scientific Camera Interfaces",
                 "SDK", "Python Toolkit", "dlls", "64_lib")
)
os.add_dll_directory(DLL_DIR)
os.environ["PATH"] = DLL_DIR + os.pathsep + os.environ["PATH"]

from thorlabs_tsi_sdk.tl_camera import TLCameraSDK
from thorlabs_tsi_sdk.tl_camera_enums import OPERATION_MODE

with TLCameraSDK() as sdk:
    cameras = sdk.discover_available_cameras()
    print(f"Cameras: {cameras}")

    with sdk.open_camera(cameras[0]) as cam:
        print(f"Camera: {cam.model} ({cam.serial_number}), "
              f"Resolution: {cam.image_width_pixels}x{cam.image_height_pixels}")

        cam.exposure_time_us = 1300
        cam.frames_per_trigger_zero_for_unlimited = 1   # 1 frame per trigger
        cam.image_poll_timeout_ms = 3000                 # 3s max per frame
        cam.operation_mode = OPERATION_MODE.SOFTWARE_TRIGGERED
        cam.arm(2)

        # flush stale frames
        while cam.get_pending_frame_or_null() is not None:
            pass
        print("Stale frames flushed")
        bias = "-" + str(0.40)
        save_dir = f"calibration/image/bias-{bias}"
        os.makedirs(save_dir, exist_ok=True)
        
        frames = np.zeros((1, 2 * M, 4 * 2 * N), dtype=np.uint8)
        phase = np.full((24, M, N), phase_levels[0], dtype=np.float32)

        plm.play()

        for i in range(16):
            phase[:, :, N//2:] = phase_levels[i]
            frames = plm.bitpack_holograms_gpu(phase)
            plm.insert_frames(frames, offset=0, format=1)
            time.sleep(1)
            cam.issue_software_trigger()
            frame = cam.get_pending_frame_or_null()
            if frame is not None:
                image = np.copy(frame.image_buffer)
                if image.max() == 0:
                    print(f"  i={i}: WARNING: Image is all zeros")
                else:
                    scaled = ((image.astype(np.float32) / image.max()) * 255).astype(np.uint8)
                    Image.fromarray(scaled).save(f"{save_dir}/value_{i}.png")
                    print(f"  i={i}: Saved")
            else:
                print(f"  i={i}: No frame received — timeout")
        
        phase[:, :, :] = phase_levels[0]
        frames = plm.bitpack_holograms_gpu(phase)
        plm.insert_frames(frames, offset=0, format=1)
        cam.disarm()

image_paths = sorted(
    glob.glob(os.path.join(save_dir, "value_*.png")),
    key=lambda p: int(p.split("_")[-1].split(".")[0])
)

phases = []
for path in image_paths:
    phase = calculate_true_phase_shift(path)
    phases.append(phase if phase >= 0 else phase + 2 * np.pi)

levels = list(range(len(phases)))

plt.figure(figsize=(8, 4))
plt.plot(levels, [p / np.pi for p in phases], 'o-', markersize=6)
plt.ylim(0, 2)
plt.xlabel("Level")
plt.ylabel("Phase shift (rad)")
plt.xticks(levels)
plt.grid(alpha=0.3)
plt.title(f"Dephase vs Level - Bias = {bias}")
plt.savefig(os.path.join(save_dir, "dephase_vs_level.png"))
plt.show()

---
## 4. Sorted Phase Plot

Plot the measured phase shifts again, this time sorted in ascending order,
to visualise the overall monotonic response of the PLM.


In [ ]:
image_paths = sorted(
    glob.glob(os.path.join(save_dir, "value_*.png")),
    key=lambda p: int(p.split("_")[-1].split(".")[0])
)
phases = []
for path in image_paths:
    phase = calculate_true_phase_shift(path)
    phases.append(phase if phase >= 0 else phase + 2 * np.pi)

levels = list(range(len(phases)))
phases = sorted(phases)
plt.figure(figsize=(8, 4))
plt.plot(levels, phases, 'o-', markersize=6)
plt.ylim(0, 2 * np.pi)
plt.xlabel("Level")
plt.ylabel("Phase shift (rad)")
plt.xticks(levels)
plt.grid(alpha=0.3)
plt.title("Dephase vs Level")
plt.show()

---
## 5. Cleanup

Stop the PLM display, close the UI window, and release the DLL resource.


In [ ]:
plm.stop()
plm.stop_ui()
plm.lib.Close()
print("Cleanup complete")